# Séance 9 — Test A/B : règles expertes contre modèle
*Session 9 — A/B testing: expert rules versus model*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/09_test_ab.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ○ démonstration |

**À la fin de cette séance, vous saurez :**

1. affecter des cartes à deux bras de façon **déterministe et stable** (hachage) ;
2. vérifier l'équilibre des bras (test A/A) ;
3. comparer deux proportions (test z, intervalle de confiance) ;
4. tenir compte de la dépendance intra-carte par un bootstrap en grappes.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("abtest")
dossier = cfg.chemin("artefacts") / "abtest"
rapport = lire_json(dossier / "rapport_regles_vs_modele.json")
print("Période :", rapport["periode"], "| unité :", rapport["unite_randomisation"])
pd.DataFrame(rapport["bras"]).T

In [ ]:
print("Décision :", rapport["decision"])
image(dossier / "figure_regles_vs_modele.png")

In [ ]:
from fraude360.abtest import experience
trafic = pd.read_csv(dossier / "trafic_regles_vs_modele.csv")
trafic.head()

### Exercice 9.1 — 🟣 Gestion + Informatique

Écrivez `affecter(cles, part_challenger, sel="fraude360")` : pour chaque clé `v`, calculez
`hashlib.md5(f"{sel}:{v}".encode()).hexdigest()`, prenez les 8 premiers caractères en base 16, modulo 10 000; si le résultat est
inférieur à `part_challenger × 10 000`, le bras est `"B_challenger"`, sinon `"A_champion"`. Renvoyez une `Series`.

In [ ]:
# VOTRE CODE ICI
import hashlib
def affecter(cles, part_challenger, sel="fraude360"):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_c = pd.Series([f"C{i:06d}" for i in range(2000)])
verifier((affecter(_c, 0.5).values == experience.affecter(_c, 0.5).values).all(), "Même affectation que le pipeline")
verifier(0.45 < (affecter(_c, 0.5) == "B_challenger").mean() < 0.55)

**❓ Question (🟢 Tous les parcours) —** Pourquoi affecter par **carte** plutôt que par transaction ? Que se passerait-il si une même carte passait d'un bras à l'autre ?

*Votre réponse ici.*

### Exercice 9.2 — 🟢 Tous les parcours

Écrivez `test_deux_proportions(succes_a, n_a, succes_b, n_b)` qui renvoie un dictionnaire avec `difference`
(p_B − p_A), `z` (variance **groupée** sous H0) et `valeur_p` (bilatérale, `scipy.stats.norm`). Appliquez-le aux taux de détection
des deux bras du rapport (`fraudes_detectees` / `fraudes_reelles`).

In [ ]:
# VOTRE CODE ICI
from scipy import stats
def test_deux_proportions(succes_a, n_a, succes_b, n_b):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
A, B = rapport["bras"]["A_champion"], rapport["bras"]["B_challenger"]
_ref = experience.comparer_deux_proportions(A["fraudes_detectees"], A["fraudes_reelles"], B["fraudes_detectees"], B["fraudes_reelles"])
_m = test_deux_proportions(A["fraudes_detectees"], A["fraudes_reelles"], B["fraudes_detectees"], B["fraudes_reelles"])
verifier_proche(_m["z"], _ref["z"], 1e-2)
verifier_proche(_m["valeur_p"], _ref["valeur_p"], 1e-3)

### Exercice 9.3 — 🔷 Informatique — approfondissement

Bootstrap en grappes : à partir de `trafic`, ré-échantillonnez les **cartes** avec remise (500 tirages) dans chaque
bras et calculez l'écart de taux de détection B − A. Donnez l'intervalle à 95 % `ic` (tuple bas, haut). Comparez à l'intervalle
« naïf » du test z.

In [ ]:
# VOTRE CODE ICI
rng = np.random.default_rng(0)
ic = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(ic[0] < ic[1] and -1 <= ic[0] <= 1)

**❓ Question (🔶 Gestion — décision d'affaires) —** Le modèle détecte plus de fraudes mais génère plus d'alertes. Rédigez la recommandation (déployer B ou non) en citant l'intervalle de confiance du coût par 1 000 transactions et le plafond d'alertes.

*Votre réponse ici.*

## Bilan

- Hachage déterministe = affectation stable, reproductible et auditable.
- Unité de randomisation = unité d'analyse : par carte, avec un bootstrap en grappes.
- La décision combine signification statistique, coût et contraintes opérationnelles.

**Pour aller plus loin :**

- `src/fraude360/abtest/experience.py` : contrôle d'équilibre, tests et décision automatisée.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*